# Imports

In [3]:
## Para CPU

import torch

# Obtenemos la versión de Torch (ej: 2.1.0)
TORCH_VERSION = torch.__version__.split('+')[0]

print(f"Instalando torch_scatter para PyTorch {TORCH_VERSION} (Versión CPU)...")

# La magia está en el '+cpu' al final de la URL
!pip install torch_scatter -f https://data.pyg.org/whl/torch-{TORCH_VERSION}+cpu.html

Instalando torch_scatter para PyTorch 2.9.0 (Versión CPU)...
Looking in links: https://data.pyg.org/whl/torch-2.9.0+cpu.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.0/108.0 kB 3.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for torch_scatter: filename=torch_scatter-2.1.2-cp312-cp312-linux_x86_64.whl size=664632 sha256=e403711398ccff35e556b14f6cd6ddeb846e6c7314e180e52787624972870aa9
  Stored in directory: /root/.cache/pip/wheels/84/20/50/44800723f57cd798630e77b3ec83bc80bd26a1e3dc3a672ef5
Successfully built torch_scatter


In [2]:
## Para GPU

# import torch

# def format_pytorch_version(version):
#     return version.split('+')[0]

# def format_cuda_version(version):
#     return 'cu' + version.replace('.', '')

# TORCH_VERSION = format_pytorch_version(torch.__version__)
# CUDA_VERSION = format_cuda_version(torch.version.cuda)

# print(f"Detectado: PyTorch {TORCH_VERSION} | CUDA {CUDA_VERSION}")

# # Instalamos usando el link específico de los binarios
# !pip install torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-{TORCH_VERSION}+{CUDA_VERSION}.html


In [ ]:
!pip install torch_geometric

In [5]:
import os
import numpy as np
import pandas as pd
import copy
import pprint
import math
import gc

import torch
import torch.nn as nn

from torch_geometric.data import Dataset
from torch_geometric.loader import DataLoader

from datetime import datetime, timezone, timedelta

from sklearn.metrics import precision_score, recall_score, f1_score, fbeta_score, average_precision_score, roc_auc_score, confusion_matrix
from scipy.special import expit # It is the optimized sigmoid of Scipy


In [6]:
# Change directory
os.chdir('/content/drive/MyDrive/nids-mitre/')

!pwd


/content/drive/MyDrive/nids-mitre


# Experiment Manager

In [7]:
class ExperimentManager:
    def __init__(self, log_file="./logs/experiments_log.csv", model_dir="./saved_models"):
        self.log_file = log_file
        self.model_dir = model_dir
        os.makedirs(os.path.dirname(log_file), exist_ok=True)
        os.makedirs(model_dir, exist_ok=True)

    def log_experiment(self, model_config=None, model_name=None, params=None, metrics=None, model_object=None):
        """
        Record an experiment in CSV format and optionally save the model.

        model_config: Recommended dict:
        - model_name (str)
        - type (str)
        - model_params (dict) # ONLY model hyperparameters
        - prob_threshold (float) # Decision threshold for probabilities (for metrics computation)
        - data_params (dict) # Optional
        - extra_params (dict) # Optional

        model_name: Name of the model (str)

        params: (legacy) Extra dict (for compatibility)
        metrics: Dict with results
        model_object: Model object (for saving)
        """

        if metrics is None:
            metrics = {}
        if params is None:
            params = {}

        # Timezone Argentina
        tz = timezone(timedelta(hours=-3))
        now = datetime.now(tz)

        # Input
        if model_config is not None:
            mname = model_config.get("model_name", model_name)
            mtype = model_config.get("type", None)
            model_params = model_config.get("model_params", {})
            threshold = model_config.get("prob_threshold", None)
            data_params = model_config.get("data_params", {})
            extra_params = model_config.get("extra_params", {})
        else:
            # legacy mode
            mname = model_name
            mtype = params.get("type", None)
            threshold = params.get("prob_threshold", None)
            model_params = params
            data_params = {}
            extra_params = {}

        # Create record
        entry = {
            "timestamp": now.strftime("%Y-%m-%d %H:%M:%S"),
            "model_name": mname,
        }
        if mtype is not None:
            entry["type"] = mtype
        if threshold is not None:
            entry["prob_threshold"] = threshold

        # Prefijos para evitar colisiones de nombres
        entry.update({f"hp_{k}": v for k, v in (model_params or {}).items()})
        entry.update({f"data_{k}": v for k, v in (data_params or {}).items()})
        entry.update({f"extra_{k}": v for k, v in {**extra_params, **params}.items()
                      if k not in ("type", "prob_threshold")})

        entry.update(metrics)

        # Save in csv
        df_new = pd.DataFrame([entry])
        if os.path.exists(self.log_file):
            df_new.to_csv(self.log_file, mode="a", header=False, index=False)
        else:
            df_new.to_csv(self.log_file, mode="w", header=True, index=False)

        print(f"Experiment recorded in {self.log_file}")

        # Save model
        if model_object is not None:
            metric_key = "AUC-PR" if "AUC-PR" in metrics else ("F1" if "F1" in metrics else None)
            metric_val = metrics.get(metric_key, 0) if metric_key else 0
            safe_key = metric_key or "metric"
            filename = f"{mname}_{now.strftime('%Y%m%d_%H%M')}_{safe_key}_{float(metric_val):.4f}"

            if "sklearn" in str(type(model_object)):
                import joblib
                joblib.dump(model_object, os.path.join(self.model_dir, f"{filename}.joblib"))
            else:
                import torch
                torch.save(model_object.state_dict(), os.path.join(self.model_dir, f"{filename}.pth"))

            print(f"Saved model: {filename}")


# Global instance
#exp_manager = ExperimentManager()

# Load data

In [8]:
class NF_IDS_Dataset(Dataset):
    def __init__(self, root_dir, split='train'):
        # root_dir: (eg: "./dataset_processed")
        # split: 'train', 'val' or 'test'
        self.split_dir = os.path.join(root_dir, split)
        super().__init__(self.split_dir)

        # List files ordered numerically to respect the time
        self.files = sorted(
            [f for f in os.listdir(self.split_dir) if f.endswith('.pt')],
            key=lambda x: int(x.split('_')[1].split('.')[0])
        )

    def len(self):
        return len(self.files)

    def get(self, idx):
        data = torch.load(
            os.path.join(self.split_dir, self.files[idx]),
            weights_only=False # to allow loading complex graph objects
        )
        return data

# Model

In [10]:
class EdgeGRU_Baseline(nn.Module):
    def __init__(self, node_dim, edge_dim, hidden_dim, output_bias_init=None):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.node_memory = {} # Memoria diccionario {global_id: hidden_state}

        # 1. ENCODER: En lugar de GNN, usamos un MLP simple para procesar la entrada
        # Entrada: Features del Nodo + Features del Edge (igual que hacíamos antes)
        # Nota: En ST-GNN la GNN mezclaba vecinos. Aquí solo miramos el propio nodo/arista.
        input_dim = (2 * node_dim) + edge_dim
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU()
        )

        # 2. MEMORIA: GRU Cell
        self.gru = nn.GRUCell(hidden_dim, hidden_dim)

        # 3. CLASIFICADOR
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

        if output_bias_init is not None:
            self.classifier[-1].bias.data.fill_(output_bias_init)

    def forward(self, x, edge_index, edge_attr, global_node_ids):
        # A. Preparar Tensores
        device = x.device
        num_nodes_batch = x.size(0)

        # B. Construir representación local (Sin mirar vecinos)
        # Usamos edge_index para saber qué nodo es origen y destino, pero no para pasar mensajes
        src, dst = edge_index
        # Concatenamos [Src_Node | Dst_Node | Edge_Attr]
        raw_features = torch.cat([x[src], x[dst], edge_attr], dim=1)

        # Codificamos a dimensión oculta
        encoded_features = self.encoder(raw_features) # [Num_Edges, Hidden]

        # C. Manejo de Memoria (Igual que ST-GNN pero operando sobre edges o nodos)
        # Para simplificar y hacerlo comparable, vamos a actualizar la memoria de los NODOS
        # usando la información de sus aristas conectadas (agregación simple sin GNN).

        # Paso 1: Recuperar memoria anterior
        hidden_states = []
        global_ids_list = global_node_ids.tolist()

        # Pre-allocate tensor for speed
        h_prev = torch.zeros(num_nodes_batch, self.hidden_dim, device=device)

        # Buscar en el diccionario (O(1) average)
        # Optimizacion: Vectorizar esto si fuera posible, pero por diccionario es seguro
        for i, gid in enumerate(global_ids_list):
            if gid in self.node_memory:
                h_prev[i] = self.node_memory[gid]

        # Paso 2: Actualizar memoria (GRU)
        # OJO: Aquí está la diferencia clave con GNN.
        # En GNN, 'x' traía info de vecinos. Aquí 'x' es estático (ones).
        # Necesitamos que la GRU reciba info de LO QUE PASÓ en las aristas.

        # Agregamos la info de las aristas a los nodos (Scatter Add/Mean)
        # Esto es "Pool" no "Convolución" (no aprende pesos de vecindad)
        from torch_scatter import scatter_mean

        # Promediamos la info de todas las aristas entrantes/salientes para cada nodo
        # para tener un vector de entrada para la GRU del nodo.
        aggr_input = scatter_mean(encoded_features, src, dim=0, dim_size=num_nodes_batch)

        # h_new = GRU(Input_Actual, Memoria_Anterior)
        h_new = self.gru(aggr_input, h_prev)

        # Paso 3: Guardar memoria
        # Detach es vital para cortar el gradiente entre snapshots (BPTT limitado)
        h_new_detached = h_new.detach()
        for i, gid in enumerate(global_ids_list):
            self.node_memory[gid] = h_new_detached[i]

        # D. Clasificación Final
        # Para clasificar la arista, usamos la memoria actualizada de sus nodos
        h_src = h_new[src]
        h_dst = h_new[dst]

        # Combinamos memorias para decidir sobre la arista
        # (Suma, promedio o concat). Usemos suma para variar.
        edge_representation = h_src + h_dst

        return self.classifier(edge_representation)


In [25]:
import torch
import torch.nn as nn

class EdgeGRU_Baseline(nn.Module):
    def __init__(self, node_dim, edge_dim, hidden_dim, output_bias_init=None):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.node_memory = {}

        # 1. ENCODER
        input_dim = (2 * node_dim) + edge_dim
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU()
        )

        # 2. GRU Cell
        self.gru = nn.GRUCell(hidden_dim, hidden_dim)

        # 3. EDGE CLASSIFIER
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

        if output_bias_init is not None:
            self.classifier[-1].bias.data.fill_(output_bias_init)

    def detach_all_memory(self):
        """
        It cuts off the flow of gradients in stored memory.
        Required for Truncated Backpropagation Through Time (TBPTT).
        """
        for k, v in self.node_memory.items():
            self.node_memory[k] = v.detach()

    def reset_memory(self):
        """Reset memory (useful when starting epoch or validation)"""
        self.node_memory = {}
    # --------------------------------------------------

    def manual_scatter_mean(self, src, index, dim_size):
        out = torch.zeros((dim_size, src.size(1)), device=src.device)
        out.index_add_(0, index, src)

        ones = torch.ones(src.size(0), 1, device=src.device)
        count = torch.zeros(dim_size, 1, device=src.device)
        count.index_add_(0, index, ones)

        count[count < 1] = 1
        return out / count

    def forward(self, x, edge_index, edge_attr, global_node_ids):
        device = x.device
        num_nodes_batch = x.size(0)

        src, dst = edge_index

        # [Src | Dst | Edge]
        raw_features = torch.cat([x[src], x[dst], edge_attr], dim=1)
        encoded_features = self.encoder(raw_features)

        # --- Memory ---
        hidden_states = []
        global_ids_list = global_node_ids.tolist()
        h_prev = torch.zeros(num_nodes_batch, self.hidden_dim, device=device)

        # Restore previous state
        for i, gid in enumerate(global_ids_list):
            if gid in self.node_memory:
                h_prev[i] = self.node_memory[gid]

        # Update GRU (using manual aggregation of incoming edges)
        aggr_input = self.manual_scatter_mean(encoded_features, src, dim_size=num_nodes_batch)
        h_new = self.gru(aggr_input, h_prev)

        # Save new state (detachment is done externally in train_epoch or here if necessary)
        # Note: Normally, we do NOT detach in the forward if we want to train BPTT.
        # Detachment is done in 'detach_all_memory' between batches.
        h_new_stored = h_new.clone()
        for i, gid in enumerate(global_ids_list):
            self.node_memory[gid] = h_new_stored[i]

        # Classify
        h_src = h_new[src]
        h_dst = h_new[dst]
        edge_representation = h_src + h_dst

        return self.classifier(edge_representation)

# Functions


## Metrics

In [11]:
def calculate_metrics_gnn(y_true, y_pred_logits, prob_threshold=0.5):
    """
    y_true: List or array of actual labels (0 or 1).
    y_pred_logits: List or array of raw model outputs (before sigmoid).
    """
    y_true = np.array(y_true)
    logits = np.array(y_pred_logits)

    # Convert logits to probabilities, and to classes (0 or 1) depend on prob_threshold
    probs = expit(logits)
    preds = (probs > prob_threshold).astype(int)

    # Threshold-dependent metrics
    prec = precision_score(y_true, preds, zero_division=0)
    rec = recall_score(y_true, preds, zero_division=0)
    f1 = f1_score(y_true, preds, zero_division=0)
    f2 = fbeta_score(y_true, preds, beta=2, zero_division=0)

    # Threshold-independent metrics
    try:
        ap = average_precision_score(y_true, probs)
        roc = roc_auc_score(y_true, probs)
    except ValueError:
        # This occurs if y_true has only one class (e.g., only benign in the batch)
        ap = 0.0
        roc = 0.5

    # Confusion matrix
    tn, fp, fn, tp = confusion_matrix(y_true, preds, labels=[0, 1]).ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0

    return {
        "Precision": prec, "Recall": rec, "F1": f1, "F2": f2, "AUC-PR": ap, "AUC-ROC": roc,
        "FPR": fpr, "TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn), "Total_Flows": len(y_true)
    }



## Train and eval

In [12]:
def train_epoch(model, loader, optimizer, criterion, device, is_temporal=False, batch_steps=10):
    model.train()
    total_loss = 0
    steps = 0  # Count of valid graphs processed (not empty)

    # Loss accumulator for Truncated Backprop
    batch_loss = 0

    # Iterate over the Loader
    # batch_idx helps to know if we are on the last element
    for batch_idx, data in enumerate(loader):
        data = data.to(device)

        # If it is an empty graph, skip
        if data.x.shape[0] == 0:
            continue

        # Forward
        if is_temporal:
            out = model(data.x, data.edge_index, data.edge_attr, data.global_node_ids)
        else:
            out = model(data.x, data.edge_index, data.edge_attr)

        # Loss calculation
        loss = criterion(out.view(-1), data.y)
        batch_loss += loss
        steps += 1

        # Backpropagation:
        # 1. Each 'batch_steps' valid steps (e.g., every 10 graphs)
        # 2. Or if it is in the LAST batch of the loader (so as not to lose the remainder)
        is_last_batch = (batch_idx == len(loader) - 1)

        if (steps > 0 and steps % batch_steps == 0) or is_last_batch:
            # Only update if there's something accumulated
            if batch_loss > 0:
                optimizer.zero_grad()
                batch_loss.backward()
                optimizer.step()

                # Reset
                total_loss += batch_loss.item()
                batch_loss = 0

                # IMPORTANT for ST-GNN: Detach memory
                if is_temporal:
                    model.detach_all_memory()

    # Average loss per valid step
    return total_loss / steps if steps > 0 else 0

@torch.no_grad()
def evaluate(model, loader, criterion, device, prob_threshold, is_temporal=False):
    model.eval()
    all_logits = []
    all_trues = []
    total_loss = 0
    steps = 0

    # Clear the memory before starting the evaluation (only if it's temporal)
    if is_temporal:
        model.node_memory = {}

    # Don't need enumerate here because we're not doing backprop
    for data in loader:
        data = data.to(device)

        if data.x.shape[0] == 0: continue

        if is_temporal:
            out = model(data.x, data.edge_index, data.edge_attr, data.global_node_ids)
        else:
            out = model(data.x, data.edge_index, data.edge_attr)

        loss = criterion(out.view(-1), data.y)
        total_loss += loss.item()

        all_logits.extend(out.view(-1).cpu().numpy())
        all_trues.extend(data.y.cpu().numpy())
        steps += 1

    metrics = calculate_metrics_gnn(all_trues, all_logits, prob_threshold)
    metrics["Loss"] = total_loss / steps if steps > 0 else 0
    return metrics

# Auxiliary

In [13]:
ROOT_PATH = "./dataset_processed"

In [14]:
# Instantiate Dataset (Only reads file names)
train_dataset = NF_IDS_Dataset(root_dir=ROOT_PATH, split='train')
val_dataset = NF_IDS_Dataset(root_dir=ROOT_PATH, split='val')

print(f"Train size: {len(train_dataset)} | Val size: {len(val_dataset)}")

# Instantiate DataLoader (Load manager)
# batch_size=1 : Important for ST-GNN to handle memory step by step
# num_workers=2 : Use 2 CPU cores to load files while training
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=False, num_workers=2, pin_memory=False) # pin_memory=False for CPU
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=2, pin_memory=False)

Train size: 1998 | Val size: 428


# Main

## First configuration

In [16]:
exp_manager = ExperimentManager(log_file="./logs/experiments_log_rnn_biasinit.csv")

In [26]:
# --- PARAMETERS ---
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EPOCHS = 60
BATCH_STEPS = 10 # backprop every 10 snapshots (sequence)
LR = 0.005

NODE_DIM = 16   # Features dummy (1s)
EDGE_DIM = 32   # 20 numeric + 7 dst_port + 5 protocol
HIDDEN_DIM = 32
DROPOUT = 0.2
BIAS_VALUE = -2.9968

POS_WEIGHT = 2.0

PROB_THRESHOLD = 0.5


print(f"Using device: {DEVICE}")


Using device: cpu


In [27]:
model_config = {
    "model_name": "EdgeGRU_BiasOn",
    "type": "GRU",
    "model_params": {
        "node_dim": NODE_DIM,
        "edge_dim": EDGE_DIM,
        "hidden_dim": HIDDEN_DIM,
        #"dropout": DROPOUT,
        "output_bias_init": BIAS_VALUE,
    },
    "prob_threshold": PROB_THRESHOLD,
    "extra_params": {
        "epochs": EPOCHS,
        "batch_steps": BATCH_STEPS,
        "pos_weight": POS_WEIGHT,
        "learning_rate": LR
    }
}


In [28]:
gru_model = EdgeGRU_Baseline(**model_config['model_params']).to(DEVICE)

criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([POS_WEIGHT]).to(DEVICE))
optimizer = torch.optim.Adam(gru_model.parameters(), lr=LR)


best_aucpr = 0.0
best_wts = copy.deepcopy(gru_model.state_dict())
best_metrics = {}

print(f"--- Starting training: {model_config['model_params']} ---")

for epoch in range(EPOCHS):
    loss = train_epoch(gru_model, train_loader, optimizer, criterion, is_temporal=True, device=DEVICE, batch_steps=BATCH_STEPS)
    metrics = evaluate(gru_model, val_loader, criterion, DEVICE, prob_threshold=PROB_THRESHOLD, is_temporal=True)

    print(f"Epoch {epoch+1} | Loss: {loss:.4f} | Val AUC-PR: {metrics['AUC-PR']:.4f} | Val F1: {metrics['F1']:.4f} | Val Rec: {metrics['Recall']:.4f}")

    if metrics['AUC-PR'] > best_aucpr:
        best_aucpr = metrics['AUC-PR']
        best_metrics = metrics
        best_wts = copy.deepcopy(gru_model.state_dict())


print(f"\nRestoring better weights (AUC-PR: {best_aucpr:.4f})...")
gru_model.load_state_dict(best_wts)
exp_manager.log_experiment(model_config=model_config, metrics=best_metrics, model_object=gru_model)

print(f"\nBest AUC-PR for EdgeGRU_Baseline obtained: {best_aucpr:.4f}")

--- Starting training: {'node_dim': 16, 'edge_dim': 32, 'hidden_dim': 32, 'output_bias_init': -2.9968} ---
Epoch 1 | Loss: 0.2248 | Val AUC-PR: 0.0458 | Val F1: 0.0000 | Val Rec: 0.0000
Epoch 2 | Loss: 0.2153 | Val AUC-PR: 0.0581 | Val F1: 0.0000 | Val Rec: 0.0000
Epoch 3 | Loss: 0.2094 | Val AUC-PR: 0.0262 | Val F1: 0.0000 | Val Rec: 0.0000
Epoch 4 | Loss: 0.2215 | Val AUC-PR: 0.0296 | Val F1: 0.0000 | Val Rec: 0.0000
Epoch 5 | Loss: 0.2182 | Val AUC-PR: 0.0372 | Val F1: 0.0000 | Val Rec: 0.0000
Epoch 6 | Loss: 0.2199 | Val AUC-PR: 0.0411 | Val F1: 0.0000 | Val Rec: 0.0000
Epoch 7 | Loss: 0.2180 | Val AUC-PR: 0.0409 | Val F1: 0.0000 | Val Rec: 0.0000
Epoch 8 | Loss: 0.2179 | Val AUC-PR: 0.0411 | Val F1: 0.0000 | Val Rec: 0.0000
Epoch 9 | Loss: 0.2178 | Val AUC-PR: 0.0410 | Val F1: 0.0000 | Val Rec: 0.0000
Epoch 10 | Loss: 0.2178 | Val AUC-PR: 0.0404 | Val F1: 0.0000 | Val Rec: 0.0000
Epoch 11 | Loss: 0.2177 | Val AUC-PR: 0.0397 | Val F1: 0.0000 | Val Rec: 0.0000
Epoch 12 | Loss: 0.217